# 19 — Detection-Aware Forecast Audit (v2.6)

**LeukemiPrediag v2.6 — outcome-free.** Re-scores the v2.4/v2.5 forecasters on `real_serial_unfiltered.csv`:

- **detection-aware error**: truth and forecast are both floored at the held-out observation's detection limit, so exact zeros no longer dominate the metric through the 1e-5 eps convention;
- **VAF-scale MAE** and **binomial NLL** of held-out alt reads (LBC only);
- **prospective ascertainment**: the primary set keeps only trajectories whose *training* window already reached VAF >= 1%, removing variants that entered the source table because of the held-out wave.

The 2% `alpha_vaf_stratum` threshold is frozen from v2.4. The v2.5 censored-growth predictions are reused unchanged. VAF forecasting only, with no clinical outcome claims. See `V26_MODEL_AUDIT.md`.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = Path("/content/drive/MyDrive")
    CANDIDATES = [MYDRIVE / "leukemia-prediagnose" / "blood-cancer-early-detection",
                  MYDRIVE / "blood-cancer-early-detection"]
    CANDIDATES += list(MYDRIVE.glob("*/blood-cancer-early-detection"))
except ImportError:
    CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p for p in CANDIDATES if (p / "src" / "forecast_scoring_v26.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"src/forecast_scoring_v26.py not found in any of: {CANDIDATES}")
os.chdir(PROJECT_ROOT)
V2 = PROJECT_ROOT / "data" / "processed" / "v2"
ARGS = [V2 / "real_serial_unfiltered.csv", V2 / "real_serial_filtered.csv",
        V2 / "growth_censored_v25" / "v25_crossfit_forecasts.csv", V2 / "forecast_scoring_v26"]
for a in ARGS[:3]:
    print(a.name, "exists:", a.exists())

In [ ]:
# ~1 min on CPU
r = subprocess.run([sys.executable, "notebooks/19_detection_aware_forecast_audit.py", *map(str, ARGS)],
                   capture_output=True, text=True)
print(r.stderr[-3000:])
r.check_returncode()
audit = json.load(open(ARGS[3] / "v26_forecast_scoring_audit.json"))
print({k: audit[k] for k in ["v25_metric_share_of_cf_error_from_zero_rows", "frac_rows_below_row_detection_limit_by_cohort", "ascertainment"]})

In [ ]:
import pandas as pd
for s in ["ALL_censored", "PRIMARY_prospective_censored"]:
    x = audit[s]
    print(f"\n== {s}: n={x['n']}, patients={x['n_patients']}")
    display(pd.DataFrame({"det_err": x["det_err"], "mae_vaf": x["mae_vaf"], "binom_nll_lbc": x["binom_nll_lbc"], "loco_det_err": x["loco_det_err"]}))
    display(pd.DataFrame(x["det_err_by_cohort"]).T)
    display(pd.DataFrame(x["bootstrap_det_err"], index=["delta", "ci_lo", "ci_hi", "p_better"]).T)
for k in ["subset:not_in_v24_dev", "subset:in_v24_dev"]:
    print(k, audit["PRIMARY_prospective_censored"][k])

## Reading the result

Negative deltas favour the first model. Acceptance still requires a patient-cluster bootstrap CI that excludes 0 against carry-forward **and** drift_global, and no loss under leave-one-cohort-out. Negative results are reported, not hidden.